# Rosette on Spider: zero-shot vs RAG few-shot vs fine-tuned NL→SQL

Three strategies for turning an English question into SQL, compared on the **Spider dev set**
(1034 questions, 20 databases unseen in Spider's training split):

| strategy | model | extra input |
|---|---|---|
| `zero_shot` | `google/flan-t5-small` | schema + question |
| `rag_fewshot` | `google/flan-t5-small` | + top-3 similar (question, SQL) pairs from Spider train, via TF-IDF |
| `fine_tuned` | `cssupport/t5-small-awesome-text-to-sql` | schema + question, in its training format |

Scored by **execution accuracy**: run the predicted and gold SQL, compare results
(ordered if the gold query has `ORDER BY`, as a multiset otherwise).

**Contamination:** ~55% of Spider dev appears verbatim in the fine-tuned checkpoint's training
data, so every number is reported on *seen*, *unseen* and *all*. Only *unseen* measures generalization.

**Kaggle:** Settings → *Accelerator: GPU T4* and *Internet: On*. Runs top to bottom in about 6 minutes on a T4.

In [1]:
# ---- config ----
import os
REPO_URL = "https://github.com/ZephyrousChimes/rosette-spider.git"
LIMIT = int(os.environ["LIMIT"]) if os.environ.get("LIMIT") else None   # None = full dev set
RECOMPUTE_CONTAMINATION = False   # True re-streams ~650MB of training data to rebuild the seen-list
BATCH_SIZE = 32

In [2]:
# ---- setup: find (or clone) the scripts ----
import subprocess, sys
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd() / "rosette-spider", Path("/kaggle/working/rosette-spider")]:
        if (p / "src" / "spider_data.py").exists():
            return p
    target = Path("/kaggle/working/rosette-spider") if Path("/kaggle").exists() else Path.cwd() / "rosette-spider"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    return target

ROOT = find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

try:
    import sentencepiece  # T5Tokenizer needs it
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)

project root: /kaggle/working/rosette-spider


In [3]:
# ---- device ----
# transformers warns that this checkpoint's embeddings are "not tied". Checked: after loading,
# lm_head, shared and both embed_tokens are bit-identical, so the warning is spurious.
import transformers
transformers.logging.set_verbosity_error()
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print("gpu:", name, "compute capability", cap)
    if cap < (7, 0):
        print("WARNING: recent PyTorch builds may not support this GPU (P100 = 6.0). Switch the accelerator to T4.")
else:
    print("WARNING: no GPU. This will work but takes ~30+ minutes on CPU.")

torch 2.10.0+cu128 | device: cuda
gpu: Tesla T4 compute capability (7, 5)


## 1. Data: download Spider dev, verify it against the official release

In [4]:
import spider_data
spider_data.download()
spider_data.verify()
dev = spider_data.load_dev()
train = spider_data.load_train()
print(f"dev: {len(dev)} questions over {len({x['db_id'] for x in dev})} databases | train (RAG bank): {len(train)}")

downloading spider_data.zip (~206MB)...
extracted dev set and 20 databases to /kaggle/working/rosette-spider/data/spider
verified: 1034/1034 dev rows identical to xlangai/spider
dev: 1034 questions over 20 databases | train (RAG bank): 7000


## 2. Contamination: which dev questions did the fine-tuned model train on?
The seen-list ships in `artifacts/spider_dev_seen.json`; set `RECOMPUTE_CONTAMINATION = True` to rebuild it from the raw training sets.

In [5]:
import contamination
if RECOMPUTE_CONTAMINATION:
    contamination.compute()
seen = contamination.load()
print(f"{len(seen)}/{len(dev)} dev questions ({len(seen)/len(dev):.1%}) appear verbatim in the fine-tuned model's training data")

566/1034 dev questions (54.7%) appear verbatim in the fine-tuned model's training data


## 3. Prompts (one example of each)

In [6]:
import strategies, metrics
if LIMIT:
    dev = dev[:LIMIT]
    print(f"LIMIT={LIMIT}: evaluating a subset")
retriever = strategies.Retriever(train)
prompts = strategies.build_prompts(dev, retriever)
for name, ps in prompts.items():
    print(f"===== {name} =====\n{ps[0]}\n")

===== zero_shot =====
Translate the question to a SQL query given this schema.
Schema:
CREATE TABLE stadium (Stadium_ID INTEGER, Location VARCHAR, Name VARCHAR, Capacity INTEGER, Highest INTEGER, Lowest INTEGER, Average INTEGER); CREATE TABLE singer (Singer_ID INTEGER, Name VARCHAR, Country VARCHAR, Song_Name VARCHAR, Song_release_year VARCHAR, Age INTEGER, Is_male VARCHAR); CREATE TABLE concert (concert_ID INTEGER, concert_Name VARCHAR, Theme VARCHAR, Stadium_ID VARCHAR, Year VARCHAR); CREATE TABLE singer_in_concert (concert_ID INTEGER, Singer_ID VARCHAR)
Question: How many singers do we have?
SQL:

===== rag_fewshot =====
Translate the question to a SQL query given this schema. Here are similar examples.
Schema:
CREATE TABLE stadium (Stadium_ID INTEGER, Location VARCHAR, Name VARCHAR, Capacity INTEGER, Highest INTEGER, Lowest INTEGER, Average INTEGER); CREATE TABLE singer (Singer_ID INTEGER, Name VARCHAR, Country VARCHAR, Song_Name VARCHAR, Song_release_year VARCHAR, Age INTEGER, Is_

## 4. Execute gold SQL, generate with each strategy, score

In [7]:
gold = [metrics.run_sql(x["db_id"], x["query"]) for x in dev]
assert all(err is None for _, err in gold), "a gold query failed to execute"

records = [{"i": i, "db_id": x["db_id"], "question": x["question"], "gold_sql": x["query"],
            "seen_in_ft_training": i in seen, "structure": metrics.structure(x["query"])}
           for i, x in enumerate(dev)]

for name, ps in prompts.items():
    preds = strategies.run_or_load(name, ps, DEVICE, BATCH_SIZE)
    metrics.score(records, name, preds, gold)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

t5-small-awesome-text-to-sql:   0%|          | 0/33 [00:00<?, ?it/s]

fine_tuned: generated 1034 in 50s


## 5. Results

In [8]:
import pandas as pd, report
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
acc = report.accuracy_table(records)
acc

,subset,n,strategy,exec_acc,ci95_lo,ci95_hi,invalid_sql,old_set_metric
0,all,1034,zero_shot,0.000,0.000,0.004,0.995,0.000
1,all,1034,rag_fewshot,0.026,0.018,0.038,0.935,0.027
2,all,1034,fine_tuned,0.101,0.084,0.120,0.732,0.101
3,unseen by fine_tuned,468,zero_shot,0.000,0.000,0.008,1.000,0.000
4,unseen by fine_tuned,468,rag_fewshot,0.017,0.009,0.033,0.940,0.019
5,unseen by fine_tuned,468,fine_tuned,0.096,0.073,0.126,0.735,0.096
6,seen by fine_tuned,566,zero_shot,0.000,0.000,0.007,0.991,0.000
7,seen by fine_tuned,566,rag_fewshot,0.034,0.022,0.052,0.931,0.034
8,seen by fine_tuned,566,fine_tuned,0.104,0.082,0.132,0.730,0.104


In [9]:
# paired exact McNemar: same questions, so compare the discordant pairs
report.paired_tests(records)

,subset,a,b,only_a_right,only_b_right,p_value
0,all,fine_tuned,rag_fewshot,101,24,0.000
1,all,fine_tuned,zero_shot,104,0,0.000
2,all,rag_fewshot,zero_shot,27,0,0.000
3,unseen by fine_tuned,fine_tuned,rag_fewshot,45,8,0.000
4,unseen by fine_tuned,fine_tuned,zero_shot,45,0,0.000
5,unseen by fine_tuned,rag_fewshot,zero_shot,8,0,0.008
6,seen by fine_tuned,fine_tuned,rag_fewshot,56,16,0.000
7,seen by fine_tuned,fine_tuned,zero_shot,59,0,0.000
8,seen by fine_tuned,rag_fewshot,zero_shot,19,0,0.000


In [10]:
# accuracy by gold-query structure
report.structure_table(records)

,subset,structure,n,zero_shot,rag_fewshot,fine_tuned
0,unseen by fine_tuned,single-table,251,0.000,0.032,0.131
1,unseen by fine_tuned,join,151,0.000,0.000,0.026
2,unseen by fine_tuned,nested/set-op,66,0.000,0.000,0.121
3,seen by fine_tuned,single-table,293,0.000,0.065,0.140
4,seen by fine_tuned,join,180,0.000,0.000,0.044
5,seen by fine_tuned,nested/set-op,93,0.000,0.000,0.108


In [11]:
# what kind of failure each strategy makes
report.failure_table(records)

,zero_shot,rag_fewshot,fine_tuned
syntax error,936,221,76
other exec error,90,22,33
"runs, wrong result",5,40,173
unknown table/column,3,724,648
correct,0,27,104


## 6. Examples: where the fine-tuned model fails on unseen questions

In [12]:
import random
random.seed(0)
fails = [r for r in records if not r["seen_in_ft_training"] and not r["fine_tuned"]["correct"]]
for r in random.sample(fails, min(8, len(fails))):
    print(f"[{r['db_id']}] {r['question']}")
    print(f"   gold: {r['gold_sql']}")
    print(f"   pred: {r['fine_tuned']['pred_sql']}")
    print(f"   kind: {report.failure_kind(r, 'fine_tuned')}\n")

[wta_1] How many matches were played in each year?
   gold: SELECT count(*) ,  YEAR FROM matches GROUP BY YEAR
   pred: SELECT year, COUNT(*) FROM matches GROUP BY year
   kind: runs, wrong result

[dog_kennels] Which professionals have operated a treatment that costs less than the average? Give me theor first names and last names.
   gold: SELECT DISTINCT T1.first_name ,  T1.last_name FROM Professionals AS T1 JOIN Treatments AS T2 WHERE cost_of_treatment  <  ( SELECT avg(cost_of_treatment) FROM Treatments )
   pred: SELECT first_name, last_name FROM Professionals AS T1 JOIN Treatments AS T2 ON T1.professional_id = T2.treatment_type_code WHERE cost_of_treatment  (SELECT AVG(cost_of_treatment) FROM Treatments)
   kind: syntax error

[student_transcripts_tracking] What are the first, middle, and last names for everybody enrolled in a Bachelors program?
   gold: SELECT DISTINCT T1.first_name ,  T1.middle_name ,  T1.last_name FROM Students AS T1 JOIN Student_Enrolment AS T2 ON T1.student_i

## 7. Save

In [13]:
import json
out = ROOT / "artifacts"
with open(out / "spider_records.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
acc.to_csv(out / "accuracy.csv", index=False)
report.paired_tests(records).to_csv(out / "paired_tests.csv", index=False)
report.structure_table(records).to_csv(out / "by_structure.csv", index=False)
report.failure_table(records).to_csv(out / "failures.csv")
print("saved to", out)

saved to /kaggle/working/rosette-spider/artifacts


## 8. Try it: ask a question, get SQL
`nl2sql.NL2SQL` wraps the fine-tuned model (the best of the three) for single queries. `src/serve.py`
serves the same function over HTTP. Give it a Spider database (`db_id`, and optionally `execute=True`),
or any `CREATE TABLE` schema text. Expect roughly 10% of answers to be right, per the table above.

In [14]:
from nl2sql import NL2SQL
nl2sql = NL2SQL(DEVICE)
for q, db in [("How many singers do we have?", "concert_singer"),
              ("What is the average age of all singers?", "concert_singer"),
              ("List the names of all teachers.", "course_teach"),
              ("How many countries are there?", "car_1")]:
    r = nl2sql.ask(q, db_id=db, execute=True)
    print(f"[{db}] {q}\n   sql:  {r['sql']}\n   rows: {r['rows'][:5] if r['rows'] is not None else None}   error: {r['error']}\n")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[concert_singer] How many singers do we have?
   sql:  SELECT COUNT(*) FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T2.Is_male = 'Singer'
   rows: None   error: OperationalError: no such column: T2.Is_male

[concert_singer] What is the average age of all singers?
   sql:  SELECT AVG(Age) FROM singer GROUP BY Name
   rows: [(52.0,), (43.0,), (29.0,), (41.0,), (32.0,)]   error: None

[course_teach] List the names of all teachers.
   sql:  SELECT Name FROM teacher WHERE Age > (SELECT MAX(Age) FROM course_arrange)
   rows: None   error: OperationalError: misuse of aggregate: MAX()

[car_1] How many countries are there?
   sql:  SELECT COUNT(*) FROM countries AS T1 JOIN car_makers AS T2 ON T1.CountryId = T2.CountryId WHERE T2.CountryName = 'Diamond'
   rows: None   error: OperationalError: no such column: T2.CountryName



In [15]:
# any schema works for generation (no database to execute against)
shop = ("CREATE TABLE customers (customer_id INTEGER, name VARCHAR, city VARCHAR, signup_date VARCHAR); "
        "CREATE TABLE orders (order_id INTEGER, customer_id INTEGER, quantity INTEGER, order_date VARCHAR)")
for q in ["How many customers are there?", "What is the total quantity ordered by customer 1?"]:
    print(q, "->", nl2sql.ask(q, schema=shop)["sql"])

How many customers are there? -> SELECT COUNT(*) FROM customers
What is the total quantity ordered by customer 1? -> SELECT SUM(quantity) FROM orders AS T1 JOIN customers AS T2 ON T1.customer_id = T2.customer_id WHERE T2.name = "1" AND T2.order_date = 1
